# Simulador de Memória — análise comparativa

Compara **First Fit**, **Best Fit** e **Worst Fit** nos workloads de `inputs/`,
nas métricas: utilização final, maior brecha residual, espaço livre total,
fusões e rejeições (fragmentação externa vs. falta de espaço).

Para cada workload: comparação tabular, **layout final da memória** por
algoritmo e gráficos de barras das métricas. No workload de fragmentação,
acrescentamos a evolução da **maior brecha** evento a evento.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt

sys.path.insert(0, str(Path.cwd()))

from simulator.parser import parse_input
from simulator.runner import run, run_simulation
from simulator.report import snapshot_summary, print_comparison
from simulator.algorithms.instantiate_algorithms import instantiate_algorithms

In [ ]:
def run_and_compare(path):
    results = run_simulation(parse_input(path))
    print_comparison([(r.algorithm, snapshot_summary(r.memory, r.stats)) for r in results])
    return results


def per_event_largest_gap(workload, choose):
    series = []
    run(workload, choose, on_event=lambda result, memory: series.append(memory.largest_gap()))
    return series

In [ ]:
_PALETTE = plt.get_cmap("tab10")
METRICAS = (
    "final_utilization_pct",
    "final_largest_gap",
    "external_fragmentation_failures",
    "no_space_failures",
)


def plot_memory_maps(results, ax):
    pids = sorted({b.pid for r in results for b in r.memory.blocks if not b.is_free})
    color = {pid: _PALETTE(i % 10) for i, pid in enumerate(pids)}
    for row, r in enumerate(results):
        for b in r.memory.blocks:
            ax.broken_barh(
                [(b.start, b.size)],
                (row - 0.4, 0.8),
                facecolors=color[b.pid] if not b.is_free else "#e6e6e6",
                edgecolor="black",
                linewidth=0.5,
            )
            if not b.is_free:
                ax.text(b.start + b.size / 2, row, b.pid, ha="center", va="center", fontsize=8)
    ax.set_yticks(range(len(results)))
    ax.set_yticklabels([r.algorithm for r in results])
    ax.set_xlabel("endereço")
    ax.set_title("Layout final da memória (cinza = brecha)")
    ax.invert_yaxis()
    ax.margins(x=0.01)


def plot_metric_bars(results, key, ax):
    names = [r.algorithm for r in results]
    values = [snapshot_summary(r.memory, r.stats)[key] for r in results]
    ax.bar(names, values, color=[_PALETTE(i % 10) for i in range(len(results))])
    ax.set_title(key)
    ax.set_ylabel(key)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", linestyle=":", alpha=0.5)


def mostra(path):
    results = run_and_compare(path)
    fig, ax = plt.subplots(figsize=(11, 3))
    plot_memory_maps(results, ax)
    fig.tight_layout()
    plt.show()
    fig, axes = plt.subplots(1, len(METRICAS), figsize=(4 * len(METRICAS), 3.2))
    for ax, key in zip(axes, METRICAS):
        plot_metric_bars(results, key, ax)
    fig.tight_layout()
    plt.show()
    return results

## Workload simples


In [ ]:
_ = mostra("inputs/workload_simples.json")

## Workload principal (`workload.json`)


In [ ]:
_ = mostra("inputs/workload.json")

## Workload de fragmentação

Além do layout e das métricas, a curva da **maior brecha** por evento mostra
a fragmentação externa surgindo: cai quando a memória se quebra em brechas
pequenas, mesmo havendo espaço livre total suficiente.


In [ ]:
results = mostra("inputs/workload_fragmentacao.json")

workload = parse_input("inputs/workload_fragmentacao.json")
fig, ax = plt.subplots(figsize=(8, 4))
for name, choose in instantiate_algorithms():
    serie = per_event_largest_gap(workload, choose)
    ax.plot(range(1, len(serie) + 1), serie, marker="o", label=name)
ax.set_xlabel("evento")
ax.set_ylabel("maior brecha")
ax.set_title("Maior brecha por evento — fragmentação externa")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## Conclusões

- **Best Fit** minimiza a sobra imediata, mas tende a pulverizar a memória em
  brechas pequenas — costuma deixar a maior brecha final menor.
- **Worst Fit** sempre usa a maior brecha, deixando sobras grandes; tende a
  manter a maior brecha final maior e a fazer mais fusões.
- **First Fit** para na primeira brecha que serve (mais barato) e fica entre os dois.
- Rejeições por **fragmentação externa** ocorrem quando o espaço livre total
  comporta o pedido, mas nenhuma brecha contígua — visível no workload de fragmentação.
